# Sovereign Contingent Claims Analysis (CCA)
## Distance-to-Default Calculation

Implementation of Gray, Merton & Bodie (2007) sovereign CCA framework.

**Inputs:**
- Exchange rate (LC per USD)
- Monetary base (local currency)
- Domestic government debt (local currency)
- Short-term external debt (USD)
- Long-term external debt (USD)
- Domestic & foreign interest rates

**Outputs:**
- Distance-to-distress (d2)
- Risk-neutral default probability
- Model-implied credit spread (bps)

In [ ]:
# =============================================================================
# SETUP
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq, fsolve
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Setup complete.")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# File paths - UPDATE THESE
INPUT_FILE = 'your_data.csv'  # <-- Your CSV file
OUTPUT_FILE = 'cca_results.csv'

# Model parameters
T = 1.0                    # Time horizon (years)
RECOVERY_RATE = 0.25       # Sovereign recovery rate (ISDA convention)
VOL_WINDOW = 52            # Rolling window for volatility (weeks)
VOL_MIN_PERIODS = 26       # Minimum periods for vol calculation

# Solver parameters
MAX_ITER = 100
TOLERANCE = 1e-6

print(f"Configuration:")
print(f"  Input file: {INPUT_FILE}")
print(f"  Time horizon: {T} year")
print(f"  Recovery rate: {RECOVERY_RATE:.0%}")
print(f"  Volatility window: {VOL_WINDOW} weeks")

---
## 1. Load and Inspect Data

In [ ]:
# Load data
df = pd.read_csv(INPUT_FILE)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded {len(df):,} rows")
print(f"Countries: {df['country'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nColumns:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# Check data coverage
print("Data coverage by column (% non-null):\n")
coverage = (df.notna().sum() / len(df) * 100).round(1)
for col, pct in coverage.items():
    print(f"  {col}: {pct}%")

In [ ]:
# Preview
df.head(10)

---
## 2. Core CCA Functions

In [ ]:
def compute_lcl_usd(monetary_base_lcu, domestic_debt_lcu, r_d, r_f, fx_rate, T=1.0):
    """
    Compute local currency liabilities in USD (Gray et al. Equation 6).
    
    LCL_$ = [(M_LC * exp(r_d * T) + B_d) * exp(-r_f * T)] / X_F
    """
    if pd.isna(monetary_base_lcu) or pd.isna(fx_rate) or fx_rate <= 0:
        return np.nan
    
    if pd.isna(domestic_debt_lcu):
        domestic_debt_lcu = 0
    if pd.isna(r_d):
        r_d = 0
    if pd.isna(r_f):
        r_f = 0
    
    lcl_lcu = (monetary_base_lcu * np.exp(r_d * T) + domestic_debt_lcu) * np.exp(-r_f * T)
    lcl_usd = lcl_lcu / fx_rate
    
    return lcl_usd


def compute_distress_barrier(st_debt, lt_debt, interest=0):
    """
    KMV distress barrier: B_f = ST + 0.5*LT + interest
    """
    if pd.isna(st_debt) and pd.isna(lt_debt):
        return np.nan
    
    st = st_debt if not pd.isna(st_debt) else 0
    lt = lt_debt if not pd.isna(lt_debt) else 0
    i = interest if not pd.isna(interest) else 0
    
    return st + 0.5 * lt + i


def black_scholes_call(A, B, r, sigma, T):
    """Black-Scholes call option value."""
    if sigma <= 0 or A <= 0 or B <= 0 or T <= 0:
        return np.nan, np.nan, np.nan
    
    d1 = (np.log(A / B) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_value = A * norm.cdf(d1) - B * np.exp(-r * T) * norm.cdf(d2)
    
    return call_value, d1, d2


def solve_merton_system(E_obs, sigma_E, B, r, T=1.0, max_iter=100, tol=1e-6):
    """
    Solve the two-equation Merton system iteratively.
    
    Eq 1: E = A*N(d1) - B*exp(-rT)*N(d2)
    Eq 2: E*sigma_E = A*sigma_A*N(d1)
    """
    result = {'A': np.nan, 'sigma_A': np.nan, 'converged': False, 'iterations': 0}
    
    if E_obs <= 0 or sigma_E <= 0 or B <= 0:
        return result
    if pd.isna(E_obs) or pd.isna(sigma_E) or pd.isna(B):
        return result
    
    # Initial guesses
    sigma_A = sigma_E * 0.5
    A = E_obs + B * np.exp(-r * T)
    
    for i in range(max_iter):
        sigma_A_old = sigma_A
        A_old = A
        
        # Step 1: Solve for A given sigma_A
        def objective_A(A_try):
            if A_try <= 0:
                return 1e10
            call_val, _, _ = black_scholes_call(A_try, B, r, sigma_A, T)
            if np.isnan(call_val):
                return 1e10
            return call_val - E_obs
        
        try:
            A_lower = max(E_obs * 0.1, 1e-10)
            A_upper = E_obs + B * 5
            A = brentq(objective_A, A_lower, A_upper, xtol=tol)
        except (ValueError, RuntimeError):
            try:
                sol = fsolve(objective_A, A_old, full_output=True)
                A = sol[0][0] if isinstance(sol[0], np.ndarray) else sol[0]
                if A <= 0:
                    return result
            except:
                return result
        
        # Step 2: Update sigma_A given A
        _, d1, _ = black_scholes_call(A, B, r, sigma_A, T)
        
        if np.isnan(d1) or norm.cdf(d1) <= 0 or A <= 0:
            return result
        
        sigma_A = (E_obs * sigma_E) / (A * norm.cdf(d1))
        sigma_A = max(0.001, min(sigma_A, 5.0))
        
        # Check convergence
        if abs(sigma_A - sigma_A_old) < tol and abs(A - A_old) / max(A_old, 1e-10) < tol:
            result = {'A': A, 'sigma_A': sigma_A, 'converged': True, 'iterations': i + 1}
            return result
    
    result = {'A': A, 'sigma_A': sigma_A, 'converged': False, 'iterations': max_iter}
    return result


def compute_cca_metrics(A, sigma_A, B, r, T=1.0, recovery=0.25):
    """Compute distance-to-distress, default probability, and spread."""
    result = {
        'distance_to_distress': np.nan,
        'default_prob': np.nan,
        'model_spread_bps': np.nan
    }
    
    if pd.isna(A) or pd.isna(sigma_A) or pd.isna(B):
        return result
    if A <= 0 or sigma_A <= 0 or B <= 0:
        return result
    
    # Distance to distress
    d2 = (np.log(A / B) + (r - 0.5 * sigma_A**2) * T) / (sigma_A * np.sqrt(T))
    result['distance_to_distress'] = d2
    
    # Default probability
    default_prob = norm.cdf(-d2)
    result['default_prob'] = default_prob
    
    # Credit spread
    if default_prob < 1.0:
        lgd = 1 - recovery
        expected_loss = lgd * default_prob
        if expected_loss < 1.0:
            spread = -np.log(1 - expected_loss) / T
            result['model_spread_bps'] = spread * 10000
    
    return result


print("CCA functions defined.")

---
## 3. Compute LCL and Volatility

In [ ]:
def get_domestic_rate(row):
    """Get domestic rate from available columns (returns decimal)."""
    for col in ['tbill_3m_pct', 'policy_rate_pct', 'money_market_rate_pct', 'govt_bond_short_pct']:
        if col in row.index and not pd.isna(row[col]):
            return row[col] / 100
    return 0.0


# Set index
df = df.set_index(['country', 'date']).sort_index()

# Compute LCL_$ for each row
print("Computing LCL_$ ...")

lcl_values = []
for idx, row in df.iterrows():
    r_d = get_domestic_rate(row)
    r_f = row.get('risk_free_rate_decimal', 0)
    if pd.isna(r_f):
        r_f = 0
    
    lcl = compute_lcl_usd(
        monetary_base_lcu=row.get('monetary_base_lcu_mn', np.nan),
        domestic_debt_lcu=row.get('gross_debt_lcu_mn', np.nan),
        r_d=r_d,
        r_f=r_f,
        fx_rate=row.get('fx_rate_per_usd', np.nan),
        T=T
    )
    lcl_values.append(lcl)

df['lcl_usd'] = lcl_values
print(f"  LCL_$ computed: {df['lcl_usd'].notna().sum():,} rows")

In [ ]:
# Compute rolling volatility by country
print("Computing volatility by country...")

countries = df.index.get_level_values('country').unique()
vol_dict = {}

for country in countries:
    country_lcl = df.loc[country, 'lcl_usd']
    
    # Log returns
    log_ret = np.log(country_lcl / country_lcl.shift(1))
    
    # Rolling vol, annualized (52 weeks)
    vol = log_ret.rolling(window=VOL_WINDOW, min_periods=VOL_MIN_PERIODS).std() * np.sqrt(52)
    
    for date, v in vol.items():
        vol_dict[(country, date)] = v

df['sigma_lcl'] = df.index.map(lambda x: vol_dict.get(x, np.nan))

print(f"  Volatility computed: {df['sigma_lcl'].notna().sum():,} rows")
print(f"  Vol range: {df['sigma_lcl'].min():.1%} to {df['sigma_lcl'].max():.1%}")

In [ ]:
# Compute distress barrier
print("Computing distress barrier...")

df['distress_barrier'] = df.apply(
    lambda row: compute_distress_barrier(
        st_debt=row.get('short_term_external_debt', np.nan),
        lt_debt=row.get('long_term_external_debt', np.nan)
    ), 
    axis=1
)

print(f"  Barrier computed: {df['distress_barrier'].notna().sum():,} rows")
print(f"  Range: ${df['distress_barrier'].min()/1e6:.1f}M to ${df['distress_barrier'].max()/1e9:.1f}B")

---
## 4. Run CCA Solver

In [ ]:
print("Running CCA solver for all rows...")
print("This may take a few minutes.\n")

results = []
total = len(df)
converged_count = 0
skipped_count = 0

for i, (idx, row) in enumerate(df.iterrows()):
    country, date = idx
    
    # Progress
    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1:,}/{total:,} ({(i+1)/total*100:.0f}%)")
    
    # Inputs
    E_obs = row['lcl_usd']
    sigma_E = row['sigma_lcl']
    B = row['distress_barrier']
    r_f = row.get('risk_free_rate_decimal', 0)
    if pd.isna(r_f):
        r_f = 0
    
    # Result template
    res = {
        'country': country,
        'date': date,
        'lcl_usd': E_obs,
        'sigma_lcl': sigma_E,
        'distress_barrier': B,
        'A_implied': np.nan,
        'sigma_A': np.nan,
        'converged': False,
        'distance_to_distress': np.nan,
        'default_prob': np.nan,
        'model_spread_bps': np.nan
    }
    
    # Skip if missing data
    if pd.isna(E_obs) or pd.isna(sigma_E) or pd.isna(B):
        skipped_count += 1
        results.append(res)
        continue
    if E_obs <= 0 or sigma_E <= 0 or B <= 0:
        skipped_count += 1
        results.append(res)
        continue
    
    # Solve Merton
    merton = solve_merton_system(E_obs, sigma_E, B, r_f, T, MAX_ITER, TOLERANCE)
    
    res['A_implied'] = merton['A']
    res['sigma_A'] = merton['sigma_A']
    res['converged'] = merton['converged']
    
    if merton['converged']:
        converged_count += 1
        
        # Compute metrics
        metrics = compute_cca_metrics(merton['A'], merton['sigma_A'], B, r_f, T, RECOVERY_RATE)
        res['distance_to_distress'] = metrics['distance_to_distress']
        res['default_prob'] = metrics['default_prob']
        res['model_spread_bps'] = metrics['model_spread_bps']
    
    results.append(res)

# Build results DataFrame
results_df = pd.DataFrame(results)
results_df['date'] = pd.to_datetime(results_df['date'])
results_df = results_df.set_index(['country', 'date']).sort_index()

print(f"\n" + "="*50)
print(f"CCA COMPLETE")
print(f"="*50)
print(f"Total rows:      {total:,}")
print(f"Converged:       {converged_count:,} ({converged_count/total*100:.1f}%)")
print(f"Skipped:         {skipped_count:,} (missing data)")

---
## 5. Results Summary

In [ ]:
# Summary statistics
valid = results_df[results_df['converged'] == True].copy()

print("\n" + "="*50)
print("RESULTS SUMMARY (converged rows only)")
print("="*50)

print(f"\nDistance to Distress (d2):")
print(f"  Mean:   {valid['distance_to_distress'].mean():.2f}")
print(f"  Median: {valid['distance_to_distress'].median():.2f}")
print(f"  Std:    {valid['distance_to_distress'].std():.2f}")
print(f"  Min:    {valid['distance_to_distress'].min():.2f}")
print(f"  Max:    {valid['distance_to_distress'].max():.2f}")

print(f"\nDefault Probability:")
print(f"  Mean:   {valid['default_prob'].mean():.2%}")
print(f"  Median: {valid['default_prob'].median():.2%}")

print(f"\nModel Spread (bps):")
spreads = valid['model_spread_bps'].dropna()
print(f"  Mean:   {spreads.mean():.0f}")
print(f"  Median: {spreads.median():.0f}")

In [ ]:
# By country
print("\nMean Distance-to-Distress by Country:")
print("-" * 40)

country_stats = valid.groupby('country').agg({
    'distance_to_distress': 'mean',
    'default_prob': 'mean',
    'model_spread_bps': 'mean'
}).round(2)

country_stats = country_stats.sort_values('distance_to_distress')
country_stats.columns = ['Avg d2', 'Avg P(default)', 'Avg Spread (bps)']

display(country_stats)

---
## 6. Visualizations

In [ ]:
# Time series of distance-to-distress for selected countries
fig, ax = plt.subplots(figsize=(14, 7))

# Select countries to plot (modify as needed)
countries_to_plot = ['Colombia', 'Russia', 'Brazil', 'Mexico', 'South Africa']
countries_to_plot = [c for c in countries_to_plot if c in valid.index.get_level_values('country')]

for country in countries_to_plot:
    try:
        country_data = valid.loc[country, 'distance_to_distress']
        ax.plot(country_data.index, country_data.values, label=country, linewidth=1.5)
    except KeyError:
        continue

ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Distress threshold')
ax.set_xlabel('Date')
ax.set_ylabel('Distance to Distress (d2)')
ax.set_title('Sovereign Distance-to-Distress Over Time')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of d2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(valid['distance_to_distress'].dropna(), bins=50, edgecolor='white', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Distance to Distress (d2)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Distance-to-Distress')

# Default probability distribution
axes[1].hist(valid['default_prob'].dropna() * 100, bins=50, edgecolor='white', alpha=0.7, color='orange')
axes[1].set_xlabel('Default Probability (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Default Probabilities')

plt.tight_layout()
plt.show()

In [ ]:
# d2 vs model spread
fig, ax = plt.subplots(figsize=(10, 6))

scatter_data = valid[['distance_to_distress', 'model_spread_bps']].dropna()
ax.scatter(scatter_data['distance_to_distress'], scatter_data['model_spread_bps'], 
           alpha=0.3, s=10)

ax.set_xlabel('Distance to Distress (d2)')
ax.set_ylabel('Model Spread (bps)')
ax.set_title('Distance-to-Distress vs Model-Implied Spread')

# Add reference line at d2 = 0
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 7. Save Results

In [ ]:
# Save to CSV
results_df.to_csv(OUTPUT_FILE)
print(f"Results saved to: {OUTPUT_FILE}")
print(f"Total rows: {len(results_df):,}")

In [ ]:
# Preview output
results_df.head(20)

---
## 8. Next Steps for Your Thesis

Now that you have the **baseline CCA distance-to-distress**, you can:

1. **Validate**: Compare `model_spread_bps` against your actual EMBI spreads
2. **Add jump dynamics**: Extend the model with oil-calibrated jumps
3. **Compare models**: Test whether jump-augmented CCA improves fit for oil exporters

Key columns for your analysis:
- `distance_to_distress`: Main risk metric (higher = safer)
- `default_prob`: Risk-neutral probability of default
- `model_spread_bps`: Model-implied CDS spread
- `A_implied`: Sovereign asset value
- `sigma_A`: Sovereign asset volatility